# IA para Cibersegurança — Setup do Projeto (Aula 1)

Notebook-base da dupla. Ao final, o **entregável de hoje** está cumprido:
ambiente configurado, repositório criado e notebook inicial versionado.

> Preencham os nomes da dupla e rodem as células na ordem.

**Dupla:** José Victor e Pedro Henrique


## 1. Montar o Google Drive
Tudo (dados, código, resultados) fica no Drive para persistir entre sessões — o runtime do Colab é volátil.

In [33]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## 2. Estrutura de pastas do projeto
Separa dado bruto de dado tratado, código, modelos e relatórios. Idempotente: rodar de novo não quebra nada.

In [34]:
from pathlib import Path

# Raiz do projeto no Drive (ajuste o nome se quiser)
PROJETO = Path('/content/drive/MyDrive/ia-ciberseguranca')
SUBDIRS = ['data/raw', 'data/processed', 'notebooks', 'src', 'models', 'reports']

for d in SUBDIRS:
    (PROJETO / d).mkdir(parents=True, exist_ok=True)

print('Projeto em:', PROJETO)
!ls -la "{PROJETO}"

Projeto em: /content/drive/MyDrive/ia-ciberseguranca
total 21
drwx------ 4 root root 4096 Aug 16 18:35 data
drwx------ 2 root root 4096 Aug 16 18:35 models
drwx------ 2 root root 4096 Aug 16 18:35 notebooks
drwx------ 2 root root 4096 Aug 16 18:35 reports
-rw------- 1 root root  136 Aug 16 18:51 requirements.txt
drwx------ 2 root root 4096 Aug 16 18:35 src


## 3. Ambiente — versões
Registra o que está instalado. Serve de diagnóstico e alimenta o `requirements.txt`.

In [35]:
import sys, platform
from importlib.metadata import version, PackageNotFoundError

print('Python :', sys.version.split()[0], '|', platform.platform())
print('-' * 40)

PACOTES = ['numpy', 'pandas', 'scikit-learn', 'scipy', 'matplotlib',
           'seaborn', 'imbalanced-learn', 'xgboost']
for p in PACOTES:
    try:
        print(f'{p:18s} {version(p)}')
    except PackageNotFoundError:
        print(f'{p:18s} (não instalado)')

Python : 3.12.13 | Linux-6.6.122+-x86_64-with-glibc2.35
----------------------------------------
numpy              2.0.2
pandas             2.2.2
scikit-learn       1.6.1
scipy              1.16.3
matplotlib         3.10.0
seaborn            0.13.2
imbalanced-learn   0.14.2
xgboost            3.3.0


## 4. Dependências das primeiras aulas
O Colab já traz numpy/pandas/sklearn/matplotlib/seaborn. Instalamos o que costuma faltar (`imbalanced-learn`, `xgboost`) — bibliotecas das aulas 2–5.

In [36]:
%pip install -q imbalanced-learn xgboost

## 5. Gerar o `requirements.txt`
Fixamos as versões **realmente instaladas** (não versões chutadas). Esse arquivo é parte do critério de reprodutibilidade da entrega — quem for corrigir precisa conseguir recriar o ambiente.

In [37]:
from importlib.metadata import version

PKGS = ['numpy', 'pandas', 'scikit-learn', 'scipy', 'matplotlib',
        'seaborn', 'imbalanced-learn', 'xgboost']

req = PROJETO / 'requirements.txt'
req.write_text('\n'.join(f'{p}=={version(p)}' for p in PKGS) + '\n')

print(req, 'gerado:\n')
print(req.read_text())

/content/drive/MyDrive/ia-ciberseguranca/requirements.txt gerado:

numpy==2.0.2
pandas==2.2.2
scikit-learn==1.6.1
scipy==1.16.3
matplotlib==3.10.0
seaborn==0.13.2
imbalanced-learn==0.14.2
xgboost==3.3.0



## 6. Reprodutibilidade — seeds
Fixa a aleatoriedade de `random`, `numpy` e, se houver, `torch`. Chamem `set_seed()` no topo de cada experimento.

In [38]:
import os, random
import numpy as np

def set_seed(seed: int = 42) -> None:
    os.environ['PYTHONHASHSEED'] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    try:
        import torch
        torch.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
    except ModuleNotFoundError:
        pass

set_seed(42)
print('Seeds fixadas em 42')

Seeds fixadas em 42


## 7. Célula-teste — mini pipeline
Confirma que o stack funciona treinando um classificador num dado sintético com desbalanceamento ~1:50 — o cenário típico de segurança. Repare na diferença entre ROC-AUC e **PR-AUC**: voltamos a isso na aula 3.

In [39]:
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import average_precision_score, roc_auc_score

X, y = make_classification(n_samples=10_000, n_features=20, n_informative=6,
                           weights=[0.98, 0.02], random_state=42)
X_tr, X_te, y_tr, y_te = train_test_split(
    X, y, test_size=0.3, stratify=y, random_state=42)

clf = RandomForestClassifier(n_estimators=200, class_weight='balanced',
                             random_state=42, n_jobs=-1).fit(X_tr, y_tr)
proba = clf.predict_proba(X_te)[:, 1]

print(f'Positivos no teste : {y_te.mean():.2%}')
print(f'ROC-AUC            : {roc_auc_score(y_te, proba):.3f}')
print(f'PR-AUC             : {average_precision_score(y_te, proba):.3f}   <- a que importa aqui')

Positivos no teste : 2.53%
ROC-AUC            : 0.791
PR-AUC             : 0.547   <- a que importa aqui


## 8. Versionar no GitHub
O jeito mais simples no Colab: **Arquivo → Salvar uma cópia no GitHub** (autentica via OAuth, escolhe o repo e faz o commit). Criem antes um repositório vazio, ex.: `ia-ciberseguranca`.

A célula abaixo só configura a identidade do Git para commits pela CLI (opcional). Não colem token no notebook — se precisarem de push por linha de comando, usem `getpass` na sessão.

In [40]:
# Opcional — identidade para commits via CLI
!git config --global user.name  "Nome Sobrenome"
!git config --global user.email "voce@cesar.school"
print('Preferência: Arquivo > Salvar uma cópia no GitHub (menu do Colab).')

Preferência: Arquivo > Salvar uma cópia no GitHub (menu do Colab).


## Checklist do entregável de hoje

- [x] Google Drive montado
- [x] Estrutura de pastas criada
- [x] Versões conferidas
- [x] `requirements.txt` gerado no projeto
- [x] Seeds fixadas
- [x] Célula-teste rodou
- [ ] Notebook versionado no GitHub

**Próxima aula:** escolher a base de dados do projeto e montar o dicionário de features.